# EEG Blink-Based Emotion Arousal Detection — Model Training

This notebook takes the blink features extracted in `01_feature_extraction.ipynb`
and trains classifiers to predict emotional state from them.

**Approach:** Start with the hard problem (27-class emotion classification),
diagnose why it doesn't work well, then reframe as a more tractable binary
arousal classification problem and compare multiple models.


## Setup

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
    print(f'Uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/blink_features_all.csv')

# Drop rows with no detected blinks — no features means no signal
df = df[df["blink_count"] > 0]

print(f"Dataset shape: {df.shape}")
df.head()

## 1. First Attempt: 27-Class Emotion Classification

The dataset is labeled with Cowen's 27-category emotion taxonomy. The first
question: can blink features alone distinguish between 27 distinct emotions?

### Handle missing values
A few inter-blink-interval features (`ibi_mean`, `ibi_std`, `ibi_entropy`,
`blink_regularity`) are undefined when a recording has only 0-1 blinks.
Two strategies were compared: median imputation vs. dropping rows.

In [ ]:
# Strategy A: median imputation
df_imputed = df.copy()
for col in ['ibi_mean', 'ibi_std', 'ibi_entropy', 'blink_regularity']:
    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

X_imp = df_imputed.drop(["emotion_label_ekman", "participant"], axis=1)
y_imp = df_imputed["emotion_label_ekman"]

scaler = StandardScaler()
X_imp_scaled = scaler.fit_transform(X_imp)

X_train, X_test, y_train, y_test = train_test_split(
    X_imp_scaled, y_imp, test_size=0.25, random_state=42, stratify=y_imp
)

rf_imputed = RandomForestClassifier(n_estimators=300, random_state=42)
rf_imputed.fit(X_train, y_train)
y_pred_imputed = rf_imputed.predict(X_test)

acc_imputed = accuracy_score(y_test, y_pred_imputed)
print(f"Random Forest accuracy (median imputation): {acc_imputed:.4f}")

In [ ]:
# Strategy B: drop rows with missing values entirely
df_no_nan = df.dropna().copy()

X_no_nan = df_no_nan.drop(["emotion_label_ekman", "participant"], axis=1)
y_no_nan = df_no_nan["emotion_label_ekman"]

scaler_no_nan = StandardScaler()
X_scaled_no_nan = scaler_no_nan.fit_transform(X_no_nan)

X_train_no_nan, X_test_no_nan, y_train_no_nan, y_test_no_nan = train_test_split(
    X_scaled_no_nan, y_no_nan, test_size=0.25, random_state=42, stratify=y_no_nan
)

rf_dropped = RandomForestClassifier(n_estimators=300, random_state=42)
rf_dropped.fit(X_train_no_nan, y_train_no_nan)
y_pred_dropped = rf_dropped.predict(X_test_no_nan)

acc_dropped = accuracy_score(y_test_no_nan, y_pred_dropped)
print(f"Random Forest accuracy (dropped NaN rows): {acc_dropped:.4f}")
print(f"\nDataset shape after dropping NaNs: {df_no_nan.shape}")

**Result: ~23-24% accuracy on 27-class classification.**

This is well above the ~3.7% random-chance baseline for 27 classes, so blink
features are picking up *some* signal — but not nearly enough to reliably
distinguish between fine-grained emotions like "admiration" vs "awe" vs
"nostalgia." This makes sense: blinks are a coarse physiological signal,
unlikely to carry that much granularity.

**Decision: reframe the problem.** Rather than 27 specific emotions, predict
a broader dimension — emotional **arousal** (calm vs activated), based on
the Circumplex Model of emotion (Russell, 1980).

## 2. Reframing as Binary Arousal Classification

In [ ]:
# Map the 27 Cowen emotion categories onto a binary arousal axis
# Labels 1-9   -> Low Arousal  (calmer emotional states)
# Labels 10-27 -> High Arousal (more activated emotional states)
arousal_mapping = {}
for i in range(1, 10):
    arousal_mapping[i] = 'Low Arousal'
for i in range(10, 28):
    arousal_mapping[i] = 'High Arousal'

y_arousal = df_no_nan["emotion_label_ekman"].map(arousal_mapping)

print("Class distribution:")
print(y_arousal.value_counts())

In [ ]:
X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(
    X_scaled_no_nan, y_arousal, test_size=0.25, random_state=42, stratify=y_arousal
)

print("X_train_binary shape:", X_train_binary.shape)
print("X_test_binary shape:", X_test_binary.shape)

## 3. Comparing Models on Binary Arousal Classification

Five classifiers trained on the same train/test split for a fair comparison.

In [ ]:
# Logistic Regression
lr_binary_model = LogisticRegression(random_state=42, max_iter=1000)
lr_binary_model.fit(X_train_binary, y_train_binary)
accuracy_lr_binary = accuracy_score(y_test_binary, lr_binary_model.predict(X_test_binary))

# Support Vector Machine
svm_binary_model = SVC(random_state=42)
svm_binary_model.fit(X_train_binary, y_train_binary)
accuracy_svm_binary = accuracy_score(y_test_binary, svm_binary_model.predict(X_test_binary))

# K-Nearest Neighbors
knn_binary_model = KNeighborsClassifier(n_neighbors=5)
knn_binary_model.fit(X_train_binary, y_train_binary)
accuracy_knn_binary = accuracy_score(y_test_binary, knn_binary_model.predict(X_test_binary))

# Random Forest
rf_binary_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_binary_model.fit(X_train_binary, y_train_binary)
accuracy_rf_binary = accuracy_score(y_test_binary, rf_binary_model.predict(X_test_binary))

# XGBoost (needs numeric labels)
le = LabelEncoder()
y_train_binary_encoded = le.fit_transform(y_train_binary)
y_test_binary_encoded = le.transform(y_test_binary)

xgb_binary_model = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_binary_model.fit(X_train_binary, y_train_binary_encoded)
accuracy_xgb_binary = accuracy_score(y_test_binary_encoded, xgb_binary_model.predict(X_test_binary))

print(f"Logistic Regression : {accuracy_lr_binary:.4f}")
print(f"SVM                 : {accuracy_svm_binary:.4f}")
print(f"K-NN                : {accuracy_knn_binary:.4f}")
print(f"Random Forest       : {accuracy_rf_binary:.4f}")
print(f"XGBoost             : {accuracy_xgb_binary:.4f}")

**Result summary:**

| Model | Accuracy |
|---|---|
| K-Nearest Neighbors | 63.1% |
| Random Forest | 64.3% |
| Logistic Regression | 64.7% |
| SVM | 65.6% |
| **XGBoost** | **68.9%** |

Reframing from 27-class to binary arousal classification roughly **tripled**
accuracy (24% → 68.9% best model). XGBoost performs best, consistent with
its strength on structured/tabular feature sets like this one.

**Note:** all reported numbers come from a single train/test split. With a
held-out test set in the low hundreds of samples, exact percentages could
shift somewhat across different random splits — the comparison between
models is more robust than any single absolute number.

## 4. Hyperparameter Tuning (XGBoost)

XGBoost was the strongest model, so it's the one worth tuning further.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

xgb_model_tune = XGBClassifier(eval_metric='logloss', random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model_tune,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_binary, y_train_binary_encoded)

print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Evaluate the tuned model on the held-out test set
best_xgb = grid_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test_binary)
test_acc_tuned = accuracy_score(y_test_binary_encoded, y_pred_tuned)
print(f"Tuned model accuracy on test set: {test_acc_tuned:.4f}")

print("\nClassification report:")
print(classification_report(y_test_binary_encoded, y_pred_tuned, target_names=le.classes_))

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(
    confusion_matrix(y_test_binary_encoded, y_pred_tuned),
    annot=True, fmt='d', cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_
)
plt.title("Confusion Matrix — Tuned XGBoost (Binary Arousal)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**Result:** Grid search found `n_estimators=300, max_depth=3,
learning_rate=0.1` as the best configuration, reaching **~70.7%
cross-validation accuracy** — a further improvement over the untuned
68.9% baseline.

## 5. Feature Importance

Which blink features actually drive the arousal prediction?

In [ ]:
feature_importances = best_xgb.feature_importances_
feature_names = X_no_nan.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

print(feature_importance_df.to_string(index=False))

plt.figure(figsize=(8,5))
sns.barplot(data=feature_importance_df, x='Importance', y='Feature', color='#4C72B0')
plt.title("Feature Importance — XGBoost Arousal Classifier")
plt.tight_layout()
plt.show()

**Result:** `blink_rate_per_min` is the single strongest predictor, by a
wide margin. This aligns with psychophysiology literature: faster, more
frequent blinking is a well-documented marker of heightened arousal, while
slower, sparser blinking is associated with calmer states. The model
recovering this exact relationship from raw EEG-derived features —
without being told it in advance — is a good sanity check that the
features are capturing real physiological signal, not noise.

## 6. Save the Final Model

Persisting the tuned model, label encoder, and feature scaler so the
pipeline can be reused outside this notebook.

In [ ]:
import joblib

joblib.dump(best_xgb, 'xgb_arousal_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(scaler_no_nan, 'feature_scaler.pkl')

print("Saved: xgb_arousal_model.pkl, label_encoder.pkl, feature_scaler.pkl")

## Summary

| Stage | Result |
|---|---|
| 27-class emotion classification (Random Forest) | ~24% accuracy |
| Binary arousal classification, best model (XGBoost) | 68.9% accuracy |
| Tuned XGBoost (GridSearchCV) | ~70.7% cross-val accuracy |
| Strongest predictive feature | `blink_rate_per_min` |

**Takeaway:** blink-derived features carry real, usable signal for coarse
emotional arousal classification, even though they're not sufficient for
fine-grained 27-class emotion recognition. The reframing from a narrow,
hard problem to a broader, tractable one — and being able to show *why*
the original approach fell short — is the core finding of this project.

**Limitations and future work are documented in the project README.**